In [ ]:
import os
from typing import TypedDict, Literal
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langgraph.types import Command
from langchain.tools import tool
from rich import print as rprint

# 构建大模型
load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {"thinking": {"type": "disabled"}}
)

# 使用代码函数模拟代替外部API调用的天气查询
@tool(parse_docstring = True)
def get_weather(city: str = "北京"):
    """
    查询指定城市的天气

    Args:
        city : 城市名称
    """
    return f"{city}的天气阴雨连绵"

@tool(parse_docstring = True)
def get_news(domain: Literal["AI", "智能体"]):
    """
    查询特定领域的当日热点

    Args:
        domain: 特定领域
    """
    if domain == "AI":
        return "gpt新版本发布"
    elif domain == "智能体":
        return "Zcode发放免费token"
    else:
        return "666"


# 绑定工具到模型
tools = [get_news, get_weather]
model_with_tool = model.bind_tools(tools = tools)

res = model_with_tool.invoke(input = "上海天气如何?")
rprint(res)